# Fantasy value backtest: did last season's ranks predict this season's value?

The board ranks players by a blend of season z-scores (`mart_board`). The natural question is
whether a ranking you could have made before the season (here, last season's full-season
z-score) predicted the value a player went on to produce. This notebook measures that for every
pair of consecutive seasons in the data, per league.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
import matplotlib.pyplot as plt
import pandas as pd
from fs import connect, describe

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
con = connect()
print(describe(con))

In [ ]:
z = con.sql('''
    select league, season, player_id, games, z_total
    from mart_category_zscores
    where window_name = 'season' and in_pool
''').df()
pairs = []
for league, part in z.groupby('league'):
    seasons = sorted(part['season'].unique())
    for prev, cur in zip(seasons, seasons[1:]):
        a = part[part['season'] == prev][['player_id', 'z_total']].rename(columns={'z_total': 'prior'})
        b = part[part['season'] == cur][['player_id', 'z_total', 'games']].rename(columns={'z_total': 'actual'})
        b = b[b['games'] >= 10]  # a handful of games is noise, not a season
        both = a.merge(b, on='player_id')
        if len(both) >= 30:
            rho = both[['prior', 'actual']].corr(method='spearman').iloc[0, 1]
            top = both.nlargest(20, 'prior')
            pairs.append({'league': league, 'prior_season': prev, 'season': cur, 'players': len(both),
                          'spearman': round(rho, 3),
                          'top20_prior_still_top20': int(top['player_id'].isin(both.nlargest(20, 'actual')['player_id']).sum())})
backtest = pd.DataFrame(pairs)
backtest

In [ ]:
if len(backtest):
    fig, ax = plt.subplots(figsize=(8, 4))
    for league, color in (('wnba', '#b23a0a'), ('nba', '#1a7f4b')):
        part = backtest[backtest['league'] == league]
        ax.plot(part['season'], part['spearman'], marker='o', label=league.upper(), color=color)
    ax.set_ylim(0, 1); ax.set_ylabel('Spearman rank correlation'); ax.legend()
    ax.set_title('How well last season\'s z-score rank predicted this season\'s')
    plt.tight_layout(); plt.show()
else:
    print('Not enough consecutive seasons in this copy of the data to backtest.')

## The biggest misses

Players whose actual value was furthest from what the prior season suggested, in the most recent
pair of seasons with at least ten games played, per league: the risers a preseason rank would have missed and the fallers it would
have overrated.

In [ ]:
rows = []
for league, part in z.groupby('league'):
    seasons = sorted(part['season'].unique())
    both = None
    for prev, cur in reversed(list(zip(seasons, seasons[1:]))):  # the latest pair with a full season behind it
        a = part[part['season'] == prev][['player_id', 'z_total']].rename(columns={'z_total': 'prior'})
        b = part[part['season'] == cur][['player_id', 'z_total', 'games']].rename(columns={'z_total': 'actual'})
        b = b[b['games'] >= 10]
        both = a.merge(b, on='player_id')
        if len(both) >= 30:
            break
        both = None
    if both is None:
        continue
    both['league'] = league; both['season'] = cur
    both['change'] = both['actual'] - both['prior']
    rows.append(both)
if rows:
    moves = pd.concat(rows)
    names = con.sql("select league, stats_player_id as player_id, full_name from dim_player where is_current").df()
    moves = moves.merge(names, on=['league', 'player_id'], how='left')
    display(moves.nlargest(10, 'change')[['league', 'season', 'full_name', 'games', 'prior', 'actual', 'change']].round(2))
    display(moves.nsmallest(10, 'change')[['league', 'season', 'full_name', 'games', 'prior', 'actual', 'change']].round(2))

## Reading it

A rank correlation around 0.6 to 0.7 is the usual finding for season-to-season fantasy value:
the prior season is a strong but far from perfect guide, and most of the error sits in minutes
(role changes, injuries), which is exactly what the board's rolling-minutes signal on the
Streamers page tries to catch early.